# Model Training

I am gonna use OHE on the remaining columns that I left in data_analysis NB, and split the data into training and testing. And then I will test a few classification models on their default parameters and then tune the parameters of the model that returns the best result. For the testing of different models and enhancement of the parameters, I am gonna use RandomizedSearchCV.

In [28]:
import pandas as pd
import numpy as np

In [29]:
df = pd.read_csv("data/cleaned_hotel_reservations_v1.csv")
df.head()

,no_of_adults,no_of_children,no_of_weekend_nights,no_of_week_nights,type_of_meal_plan,required_car_parking_space,room_type_reserved,lead_time,market_segment_type,repeated_guest,no_of_previous_cancellations,no_of_previous_bookings_not_canceled,avg_price_per_room,no_of_special_requests,booking_status
0,2,0,1,2,Meal Plan 1,0,Room_Type 1,224,Offline,0,0,0,65.00,0,0
1,2,0,2,3,Not Selected,0,Room_Type 1,5,Online,0,0,0,106.68,1,0
2,1,0,2,1,Meal Plan 1,0,Room_Type 1,1,Online,0,0,0,60.00,0,1
3,2,0,0,2,Meal Plan 1,0,Room_Type 1,211,Online,0,0,0,100.00,0,1
4,2,0,1,1,Not Selected,0,Room_Type 1,48,Online,0,0,0,94.50,0,1


In [30]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 25395 entries, 0 to 25394
Data columns (total 15 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   no_of_adults                          25395 non-null  int64  
 1   no_of_children                        25395 non-null  int64  
 2   no_of_weekend_nights                  25395 non-null  int64  
 3   no_of_week_nights                     25395 non-null  int64  
 4   type_of_meal_plan                     25395 non-null  str    
 5   required_car_parking_space            25395 non-null  int64  
 6   room_type_reserved                    25395 non-null  str    
 7   lead_time                             25395 non-null  int64  
 8   market_segment_type                   25395 non-null  str    
 9   repeated_guest                        25395 non-null  int64  
 10  no_of_previous_cancellations          25395 non-null  int64  
 11  no_of_previous_bookings_no

---

## Preprocessing

In [31]:
print(df["type_of_meal_plan"].unique().tolist())
print(df["room_type_reserved"].unique().tolist())
print(df["market_segment_type"].unique().tolist())

['Meal Plan 1', 'Not Selected', 'Meal Plan 2', 'Meal Plan 3']
['Room_Type 1', 'Room_Type 4', 'Room_Type 6', 'Room_Type 5', 'Room_Type 2', 'Room_Type 7', 'Room_Type 3']
['Offline', 'Online', 'Corporate', 'Aviation', 'Complementary']


In [32]:
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(sparse_output=False, drop="first")
encoded = ohe.fit_transform(df[["type_of_meal_plan", "room_type_reserved", "market_segment_type"]])
encoded_df = pd.DataFrame(encoded, columns = ohe.get_feature_names_out(["type_of_meal_plan", "room_type_reserved", "market_segment_type"]))
encoded_df.head(2)

,type_of_meal_plan_Meal Plan 2,type_of_meal_plan_Meal Plan 3,type_of_meal_plan_Not Selected,room_type_reserved_Room_Type 2,room_type_reserved_Room_Type 3,room_type_reserved_Room_Type 4,room_type_reserved_Room_Type 5,room_type_reserved_Room_Type 6,room_type_reserved_Room_Type 7,market_segment_type_Complementary,market_segment_type_Corporate,market_segment_type_Offline,market_segment_type_Online
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [33]:
df = pd.concat([df, encoded_df], axis = 1)
df.head(1)

,no_of_adults,no_of_children,no_of_weekend_nights,no_of_week_nights,type_of_meal_plan,required_car_parking_space,room_type_reserved,lead_time,market_segment_type,repeated_guest,...,room_type_reserved_Room_Type 2,room_type_reserved_Room_Type 3,room_type_reserved_Room_Type 4,room_type_reserved_Room_Type 5,room_type_reserved_Room_Type 6,room_type_reserved_Room_Type 7,market_segment_type_Complementary,market_segment_type_Corporate,market_segment_type_Offline,market_segment_type_Online
0,2,0,1,2,Meal Plan 1,0,Room_Type 1,224,Offline,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [34]:
df = df.drop(columns = ["type_of_meal_plan", "room_type_reserved", "market_segment_type"])
df["booking_status"] = df.pop("booking_status")
df.head(1)

,no_of_adults,no_of_children,no_of_weekend_nights,no_of_week_nights,required_car_parking_space,lead_time,repeated_guest,no_of_previous_cancellations,no_of_previous_bookings_not_canceled,avg_price_per_room,...,room_type_reserved_Room_Type 3,room_type_reserved_Room_Type 4,room_type_reserved_Room_Type 5,room_type_reserved_Room_Type 6,room_type_reserved_Room_Type 7,market_segment_type_Complementary,market_segment_type_Corporate,market_segment_type_Offline,market_segment_type_Online,booking_status
0,2,0,1,2,0,224,0,0,0,65.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0


---

## Saving the Preprocessed Dataset

In [35]:
df.to_csv("data/ohe_hotel_res_data.csv", index = False)

---

## Train/Test Split

In [36]:
from sklearn.model_selection import train_test_split

X = df.drop(columns = "booking_status")
y = df["booking_status"]
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8)
print(X_train.shape, X_test.shape)

(20316, 24) (5079, 24)


---

## Finding the Best Model

As it is a classification problem, I am gonna use Logistic Regression, Random Forest, & Decision Tree Classifiers.

In [37]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [38]:
paramters = {
    "logisticReg" : {"model" : LogisticRegression(max_iter=5000), "params" : {}},
    "decTreeClf" : {"model" : DecisionTreeClassifier(), "params" : {}},
    "randForestClf" : {"model" : RandomForestClassifier(), "params" : {}},
}

In [39]:
from sklearn.model_selection import RandomizedSearchCV

scores = []
for model_name, model_params in paramters.items():
    randomSCV = RandomizedSearchCV(model_params["model"], model_params["params"], cv = 8, 
                                   n_iter=20, random_state=42, n_jobs=-1) #n_jobs = -1 uses all the available cpu cores
    randomSCV.fit(X_train, y_train)

    scores.append({
        "model" : model_name,
        "best_params" : randomSCV.best_params_,
        "best_score" : randomSCV.best_score_
    }
    )

scores_df = pd.DataFrame(scores)

c:\Users\Ahmad Maqsood\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 1 is smaller than n_iter=20. Running 1 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
c:\Users\Ahmad Maqsood\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 1 is smaller than n_iter=20. Running 1 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
c:\Users\Ahmad Maqsood\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 1 is smaller than n_iter=20. Running 1 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


In [40]:
scores_df

,model,best_params,best_score
0,logisticReg,{},0.807049
1,decTreeClf,{},0.787213
2,randForestClf,{},0.827280


Random Forest Classifier is performing the best for this dataset. Let's optimize its paramters.

---

## Random Forest Classifier(Parameter Tuning)

In [41]:
parameters_rfc = {
    'n_estimators': [50, 100, 200, 300],
    'max_depth': [None, 10, 20, 30],            
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],               
    'bootstrap': [True, False]                  
}

randomSCV = RandomizedSearchCV(RandomForestClassifier(), parameters_rfc, 
                               cv = 8, n_iter=20, n_jobs=-1, random_state=42) # n_jobs = -1 uses all the available cpu cores
randomSCV.fit(X_train, y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestClassifier()
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'bootstrap': [True, False], 'max_depth': [None, 10, ...], 'min_samples_leaf': [1, 2, ...], 'min_samples_split': [2, 5, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",20
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionch

In [42]:
print(f"Best Paramters of Random Forest Classifier after parameter Tuning : {randomSCV.best_params_}")
print(f"Best Score of Random Forest Classifier after Parameter Tuning : {randomSCV.best_score_}")

Best Paramters of Random Forest Classifier after parameter Tuning : {'n_estimators': 100, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_depth': 20, 'bootstrap': False}
Best Score of Random Forest Classifier after Parameter Tuning : 0.8448520016870675


Our model score improved from 0.82 to 0.84. Let's train and test the model on unseen data.

---

## Random Forest Classifier

In [43]:
model = RandomForestClassifier(n_estimators=100, min_samples_split=10, min_samples_leaf=2, max_depth=20, bootstrap=False, random_state=42)
model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",20
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",10
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",False
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [44]:
print(f"Model accuracy score on unseen data : {model.score(X_test, y_test)}")

Model accuracy score on unseen data : 0.8499704666272888


In [45]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.87      0.93      0.90      3626
           1       0.78      0.66      0.72      1453

    accuracy                           0.85      5079
   macro avg       0.83      0.79      0.81      5079
weighted avg       0.85      0.85      0.85      5079



The report indicates that the model is performing quite well overall, with an 85% accuracy, but it is significantly more reliable at identifying non-cancellations (Class 0) than cancellations (Class 1).

---

## Saving the Model, OHE, & Feature Names

In [47]:
import joblib

joblib.dump(model, "model/random_forest_classifier.pkl")
joblib.dump(ohe, "model/one_hot_encoder.pkl")
joblib.dump(X.columns.tolist(), "model/feature_columns.pkl")

['model/feature_columns.pkl']